# Chapter 1: Getting Started with Flask 🌶️

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <span style='color:#aaa'>← (Start of Handbook)</span>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='2. routing_and_url_building.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 2: Routing →</a>
</div>

---

## 🎯 The Big Picture

Imagine you’ve just built a machine-learning model that predicts house prices.
It works perfectly on your laptop — but your client wants to type in a house description
and see the prediction in their browser, from anywhere in the world.
**Flask is the simplest possible way to build that in Python.**
This chapter sets up your Flask kitchen so you can start cooking web apps.

---

## 🧠 Core Concepts

### What is a web framework?

When your browser visits a URL, a lot happens behind the scenes. A **web framework** handles
the boring, repetitive parts so you can focus on your app’s logic. Here’s the journey of one request:

1. **Browser** sends an HTTP request (`GET /`)
2. **Operating system** receives it on a TCP port (usually 5000 in development)
3. **WSGI server** (see below) translates raw bytes → Python dict
4. **Flask** matches the URL to your Python function (**routing**)
5. **Your function** runs and returns a response (HTML, JSON, …)
6. **Flask** wraps the response in proper HTTP headers and sends it back

---

### Django vs Flask

| Feature | Django 🏰 | Flask 🔪 |
|---|---|---|
| Analogy | Full kitchen — oven, fridge, and plates included | A chef’s knife — sharp, precise, nothing extra |
| ORM | Built-in | You choose (SQLAlchemy, Peewee, …) |
| Admin panel | Auto-generated | Plugin or DIY |
| Learning curve | Steeper | Gentle |
| Best for | Large CRUD apps | APIs, micro-services, ML inference |

**Rule of thumb:** reach for Flask when you want control; reach for Django when you want batteries.

---
### The Flask stack

```text
+----------------------------------+
|  Your Code  (routes, logic)      |  <- you write this
+----------------------------------+
|  Jinja2     (HTML templates)     |  <- bundled with Flask
+----------------------------------+
|  Werkzeug   (HTTP utilities)     |  <- bundled with Flask
+----------------------------------+
|  WSGI interface                  |  <- the contract between server & app
+----------------------------------+
```

---
### What is WSGI?

**WSGI** (Web Server Gateway Interface, PEP 3333) is a standard *contract* between a Python
web app and the server that runs it — think of it like an electrical socket:
your appliance (Flask) just needs to match the socket shape (WSGI), and it works
with any compliant server (Gunicorn, uWSGI, Waitress, …).

---

## ⚡ Syntax & First Usage

### Installation

A **virtual environment** is an isolated Python installation for your project — it keeps your project's packages separate from your system Python and from other projects. Without one, every project would share the same global packages, leading to version conflicts. Always create one before installing project dependencies.

Flask lives on PyPI. Install it inside a virtual environment:

```bash
python -m venv .venv          # create isolated environment
source .venv/bin/activate      # macOS / Linux
# .venv\Scripts\activate      # Windows

pip install flask
```

After installation, the `flask` package (and its two main dependencies — **Werkzeug** and **Jinja2**) will be available.

In [ ]:
import flask
print("Flask version:", flask.__version__)

import werkzeug, jinja2
print("Werkzeug version:", werkzeug.__version__)
print("Jinja2 version:", jinja2.__version__)

### The Minimal Flask App

Every Flask application starts with **four lines of structural code**:

1. **Import** the `Flask` class
2. **Instantiate** the app object
3. **Decorate** a function to attach it to a URL
4. **Return** a response string from that function

> 💡 **What is a decorator?** A decorator is Python syntax using `@` that wraps a function with extra behavior — without changing the function itself. `@app.route('/')` tells Flask: *"Register the function below for the URL `/`.*" You'll see decorators constantly in Flask code.

The cell below shows the minimal app with a comment on *every* line so nothing is magic.

In [ ]:
from flask import Flask          # Import the Flask class

app = Flask(__name__)            # Create app; __name__ tells Flask where resources are

@app.route('/')                  # decorator: register the URL '/' to the function below
def index():
    return 'Hello, World!'       # return a plain string -> Flask wraps it in HTTP 200

if __name__ == '__main__':
    app.run(debug=True)          # start dev server; NEVER use debug=True in production

# Inspect the app object
print("App name   :", app.name)
print("Debug mode :", app.debug)
print("Root path  :", app.root_path)

> 💡 **Tip — `__name__` explained:**
> When Python runs a file directly, `__name__` equals `'__main__'`.
> When imported as a module, `__name__` equals the module name (e.g. `'app'`).
> Flask uses this string to locate templates and static files relative to your project.

> ⚠️ **Warning — `debug=True`:**
> The debug server auto-reloads on code changes and shows interactive tracebacks in the browser.
> Those tracebacks let *anyone* run arbitrary Python on your machine.
> **Always set `debug=False` (the default) in production.**

### Two Ways to Run Your App

**Option A — run the file directly**
```bash
python app.py          # runs app.run(debug=True) from the if __name__ == '__main__' block
```

**Option B — use the Flask CLI** *(recommended)*
```bash
export FLASK_APP=app          # tell Flask which module contains the app
export FLASK_DEBUG=1          # enable debug/reload (equivalent to debug=True)
flask run                     # starts on http://127.0.0.1:5000 by default
flask run --host=0.0.0.0 --port=8080   # custom host/port
```

The CLI is preferred because it separates *configuration* (env vars) from *code*, which
makes it easier to change settings between development, staging, and production.

### Testing Without a Real Server

Flask ships with a **test client** that lets you send fake HTTP requests to your app
inside a Jupyter cell — no browser, no running server needed.
We’ll use this pattern throughout the handbook to verify behaviour instantly.

In [ ]:
from flask import Flask

app = Flask(__name__)

@app.route('/')
def index():
    return 'Hello, World!'

# Use the built-in test client
with app.test_client() as client:
    response = client.get('/')
    print("Status code :", response.status_code)
    print("Body text   :", response.data.decode())
    print("Content-Type:", response.content_type)

---

## 🔬 Deep Dive

### ⚖️ `app.run()` vs `flask run`

Both launch a development server, but they differ in ergonomics and flexibility.

In [ ]:
# app.run() - embedded in the Python file
# PROS: Easy to get started - just `python app.py`; good for demos and notebooks
# CONS: Debug flag is hardcoded in source; env vars not respected

# flask run - CLI entry point
# PROS: Reads FLASK_APP, FLASK_DEBUG, FLASK_RUN_PORT from environment
#       Clean separation of config from code; easy to switch dev/prod settings
# CONS: Requires exporting env vars (or a .flaskenv file)

from flask import Flask

demo_app = Flask('demo')

@demo_app.route('/ping')
def ping():
    return 'pong'

with demo_app.test_client() as c:
    r = c.get('/ping')
    print("flask run equivalent result:", r.data.decode())

### Multiple Routes

Real apps have many pages. Each `@app.route()` decorator maps a URL pattern to a **view function** —
Flask’s term for the Python function that handles a request.

In [ ]:
from flask import Flask

app = Flask(__name__)

@app.route('/')
def home():
    return '<h1>Home Page</h1>'

@app.route('/about')
def about():
    return '<h1>About Page</h1><p>Flask handbook demo.</p>'

@app.route('/greet/<name>')     # <name> is a URL variable - covered in Chapter 2
def greet(name):
    return f'Hello, {name}!'

# Verify all three routes
with app.test_client() as client:
    for url in ['/', '/about', '/greet/Ada']:
        r = client.get(url)
        print(f"GET {url:<20} -> {r.status_code}  {r.data.decode()[:60]}")

### Response Types

A Flask view function can return several things.
Understanding the difference helps you build APIs *and* HTML pages with the same framework.

In [ ]:
from flask import Flask, jsonify, make_response

app = Flask(__name__)

# 1. Plain string -> 200 text/html
@app.route('/plain')
def plain():
    return 'Just a string'

# 2. Tuple (body, status_code) -> custom status
@app.route('/not-found')
def not_found():
    return 'Page not found', 404

# 3. jsonify -> Content-Type: application/json
@app.route('/json')
def json_response():
    return jsonify({'message': 'Hello', 'version': 1})

# 4. make_response -> full control over headers
@app.route('/custom')
def custom():
    resp = make_response('<b>Custom!</b>', 200)
    resp.headers['X-Powered-By'] = 'Flask Handbook'
    return resp

with app.test_client() as c:
    for url in ['/plain', '/not-found', '/json', '/custom']:
        r = c.get(url)
        ct = r.content_type.split(';')[0]
        body = r.data.decode()[:50]
        print(f"GET {url:<12} -> {r.status_code}  [{ct}]  {body}")

### Project Structure

Flask is unopinionated — you can organise your files however you like.
In practice, three layouts cover 95% of projects:

In [ ]:
small = '''
my_app/
    app.py          # routes + app factory
    templates/      # Jinja2 HTML files
        index.html
    static/         # CSS, JS, images
        style.css
'''

medium = '''
my_app/
    app/
        __init__.py     # create_app() factory
        routes.py       # view functions
        models.py       # database models
        templates/
        static/
    config.py           # DevelopmentConfig / ProductionConfig
    run.py              # entry point
'''

large = '''
my_app/
    app/
        __init__.py
        auth/           # Blueprint: login, register, logout
            __init__.py
            routes.py
        blog/           # Blueprint: posts, categories
            __init__.py
            routes.py
        templates/
        static/
    tests/
    config.py
    run.py
'''

for label, layout in [('Small', small), ('Medium', medium), ('Large', large)]:
    print('-' * 50)
    print(f' {label} app layout')
    print(layout)


### Context Globals: `current_app`, `g`, `request`, `session`

Flask uses **context locals** — thread-safe proxies that point to the *current* request or
application without you having to pass them around as arguments.

Flask uses two **context** types to make objects available to your code without passing them as arguments:

- **Application context** — active whenever Flask is processing anything (startup, requests, CLI commands). Provides access to the app configuration and `g` (per-request scratch space).
- **Request context** — pushed on top of the app context for every incoming HTTP request. Provides `request` (the current HTTP request) and `session` (the user's signed cookie data).

| Object | Context | What it holds |
|---|---|---|
| `current_app` | Application | The active Flask app instance |
| `g` | Application | Per-request scratch space (reset each request) |
| `request` | Request | Incoming HTTP data (method, form, JSON, …) |
| `session` | Request | Signed cookie-based user session |

> 💡 **Tip:** These only exist *inside* a request (or inside an `app_context` / `test_request_context`).
> Accessing them outside raises a `RuntimeError`.

In [ ]:
from flask import Flask, current_app, g, request

app = Flask(__name__)

# 1. Application context (no HTTP request needed)
with app.app_context():
    print("current_app.name :", current_app.name)
    g.user = 'Ada'
    print("g.user           :", g.user)

# 2. Request context (simulates an incoming HTTP request)
with app.test_request_context('/search?q=flask', method='GET'):
    print("\nrequest.path     :", request.path)
    print("request.method   :", request.method)
    print("request.args['q']:", request.args.get('q'))

# 3. Inside a real view via test_client
@app.route('/context-demo')
def context_demo():
    info = {'app': current_app.name, 'method': request.method, 'path': request.path}
    return str(info)

with app.test_client() as c:
    r = c.get('/context-demo')
    print("\nView response    :", r.data.decode())

---

## 🧪 What If? Experimentation

Good engineers ask “what breaks?” before it breaks in production.
Let’s deliberately trigger three common beginner mistakes.

### ❓ Q1: What if you forget `debug=True`?

When `debug=False` (the default), Flask **suppresses** the full traceback and returns
a generic 500 page instead. The error is still logged to the console, but users (and you
in the browser) see nothing useful.
In development you almost always want `debug=True` — just never ship it that way.

In [ ]:
from flask import Flask

def make_buggy_app(debug_flag):
    a = Flask(__name__)
    a.config['TESTING'] = True
    a.debug = debug_flag

    @a.route('/oops')
    def oops():
        raise ValueError("Something went wrong!")

    return a

# debug=False: exception is swallowed, returns 500
app_prod = make_buggy_app(debug_flag=False)
with app_prod.test_client() as c:
    r = c.get('/oops')
    print(f"debug=False -> status {r.status_code}")

# debug=True + PROPAGATE_EXCEPTIONS: exception surfaces so you can see it
app_dev = make_buggy_app(debug_flag=True)
app_dev.config['PROPAGATE_EXCEPTIONS'] = True
try:
    with app_dev.test_client() as c:
        c.get('/oops')
except ValueError as e:
    print(f"debug=True  -> exception surfaced: {e}")

### ❓ Q2: What if two routes share the same URL?

Flask stores routes in a `Map` object. If you try to register the *same URL pattern* twice
with **different endpoint names**, Flask raises an `AssertionError` at startup —
better to crash loudly at boot than silently serve the wrong handler.

In [ ]:
from flask import Flask

app = Flask(__name__)

@app.route('/clash')
def handler_one():
    return 'Handler One'

try:
    @app.route('/clash')          # same URL, different function name
    def handler_two():
        return 'Handler Two'
    print("No error raised (this should not print)")
except AssertionError as e:
    print(f"AssertionError caught: {e}")

# The original route still works
with app.test_client() as c:
    r = c.get('/clash')
    print("Winner:", r.data.decode())

### ❓ Q3: What if you pass the wrong string to `Flask(__name__)`?

`Flask(__name__)` uses the module name to calculate `root_path` — the directory where
Flask looks for your `templates/` and `static/` folders.
Passing a wrong string won’t crash Flask immediately, but it will silently mis-locate
your assets and lead to confusing 404 errors for templates and static files.

In [ ]:
from flask import Flask
import os

app_correct = Flask(__name__)
print("Correct  root_path:", app_correct.root_path)

app_wrong = Flask('nonexistent_package')
print("Wrong    root_path:", app_wrong.root_path)

print("\nTemplates would be looked up at:")
print("  Correct :", os.path.join(app_correct.root_path, 'templates'))
print("  Wrong   :", os.path.join(app_wrong.root_path, 'templates'))
print("\n-> Always use Flask(__name__) unless you know exactly what you're doing.")

---

## 🚀 Real-World Project Link

Every chapter in this handbook builds toward our capstone — a **full-stack Flask Blog**.

**Chapter 1** gives us the skeleton:

- The `Flask(__name__)` app object
- The development server (`flask run`)
- The project structure we’ll scale through the rest of the handbook

Here’s what our blog’s `app/__init__.py` will eventually look like
(we’re just sketching it now — every piece will be explained in the chapters ahead):

```python
from flask import Flask
from .config import config

def create_app(config_name='development'):
    app = Flask(__name__)
    app.config.from_object(config[config_name])

    # Register blueprints (Chapter 8)
    from .blog import blog as blog_blueprint
    app.register_blueprint(blog_blueprint)

    # Init extensions: SQLAlchemy, Login, Mail ... (Chapters 5-7)

    return app
```

By the end of the handbook you’ll be able to read every line of that and know exactly why it’s there.

---

## 📋 Chapter Summary & Cheat Sheet

| Concept | One-liner |
|---|---|
| Flask | Micro web framework — routes HTTP requests to Python functions |
| WSGI | Standard interface between Python apps and web servers |
| `Flask(__name__)` | Creates the app; `__name__` anchors file lookup |
| `@app.route('/')` | Decorator that maps a URL to a view function |
| `app.run(debug=True)` | Dev server — reload + tracebacks; **never in prod** |
| `flask run` | CLI launcher; reads config from env vars |
| `app.test_client()` | In-process HTTP client for testing |
| `current_app` | Proxy to the active app inside a context |
| `request` | Proxy to the current HTTP request |
| `g` | Per-request scratch pad |

Copy the cheat sheet below into your own notes:

In [ ]:
# ================================================
#  Flask Chapter 1 - Copy-Paste Cheat Sheet
# ================================================

from flask import (
    Flask,           # app factory
    request,         # current HTTP request proxy
    current_app,     # current app proxy
    g,               # per-request scratch space
    jsonify,         # dict -> JSON response
    make_response,   # build a response manually
    redirect,        # 301/302 redirect
    url_for,         # reverse URL lookup (Chapter 2)
)

app = Flask(__name__)

@app.route('/')
def index():
    return 'Hello!'

@app.route('/user/<username>')
def profile(username):
    return f'Profile: {username}'

@app.route('/api/status')
def status():
    return jsonify({'ok': True})

@app.route('/forbidden')
def forbidden():
    return 'Forbidden', 403

with app.test_client() as c:
    print(c.get('/').data.decode())
    print(c.get('/user/Ada').data.decode())
    print(c.get('/api/status').data.decode())
    print(c.get('/forbidden').status_code)

---

## 💪 Practice Prompts

Work through these on your own before moving to Chapter 2.
There are no “right” answers — experiment freely!

---

### 🏋️ Challenge 1: Multi-Page Mini-Site

Build a Flask app with **at least four routes**:

- `/` — home page with your name and a short bio
- `/projects` — a list of 3 projects you’ve worked on (or made up)
- `/contact` — email/social links
- `/404-demo` — a route that intentionally returns a 404 status code

Use `make_response` or a status-code tuple where appropriate.
Test every route with `app.test_client()`.

---

### 🏋️ Challenge 2: Dynamic Greeting with URL Variables

Create a route `/greet/<name>/<language>` that returns a greeting in the
requested language. Support at least: `en` (English), `es` (Spanish), `fr` (French).
Return a 400 response with a helpful message if an unsupported language is passed.

*Hint: you haven’t learned URL converters yet — a simple `if/elif` works fine.*

---
### 🏋️ Challenge 3: Request Inspector

Build a `/inspect` route that returns a plain-text or JSON summary of everything
Flask can tell you about the incoming request:

- HTTP method
- Full URL
- All query-string parameters
- All request headers

Use `app.test_request_context()` or the test client with custom headers
(`c.get('/inspect?foo=bar', headers={'X-Custom': 'hello'})`).

> 💡 **Tip:** `request.headers` is a dict-like object — try iterating over it with `dict(request.headers)`.

### 🧑‍💻 Your Sandbox

Use the cell below to work on the challenges above. All the imports you need are already there.

In [ ]:
# Your sandbox
from flask import Flask, request, jsonify, make_response

app = Flask(__name__)

# Start coding here



# Test your routes
with app.test_client() as c:
    pass  # replace with your test calls

---

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <span style='color:#aaa'>← (Start of Handbook)</span>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='2. routing_and_url_building.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 2: Routing →</a>
</div>

*Flask Handbook — Chapter 1 · Built with ❤️ and Jinja2*

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <span style='color:gray; font-size:1.05em;'>Previous</span>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='2. routing_and_url_building.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>